Load the feature attributions

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

path = Path("results")

df = pd.read_csv(path / "200_feature_attributions.csv", dtype={"ig_token": str})

df["ig_token"] = df["ig_token"].fillna("[EMPTY]")

explainer_result_cols = [
    "sv_value",
    "ig_value",
    "gpt_index_value",
    "gpt_index_word_value",
]
print("Original DataFrame:")
display(df.head())

df["gpt_index_value"] = df["gpt_index_value"] * -1
df["gpt_index_word_value"] = df["gpt_index_word_value"] * -1

print("DataFrame after sign inversion:")
display(df.head())

Original DataFrame:


,Unnamed: 0,id,dataset,sv_value,sv_token,sv_base_value,ig_value,ig_token,ig_base_value,gpt_index_value,gpt_index_word_value
0,670470,30685546,train,-0.002235,Wireless,-0.843256,-0.044053,wireless,-1.049297,0.0035,0.015
1,670471,30685546,train,0.005878,and,-0.843256,-0.003780,and,-1.049297,0.0028,0.005
2,670472,30685546,train,-0.000972,continuous,-0.843256,-0.004581,continuous,-1.049297,0.0023,0.008
3,670473,30685546,train,-0.000972,monitoring,-0.843256,0.002719,monitoring,-1.049297,0.0031,0.002
4,670474,30685546,train,-0.000972,of,-0.843256,0.010749,of,-1.049297,0.0027,0.012


DataFrame after sign inversion:


,Unnamed: 0,id,dataset,sv_value,sv_token,sv_base_value,ig_value,ig_token,ig_base_value,gpt_index_value,gpt_index_word_value
0,670470,30685546,train,-0.002235,Wireless,-0.843256,-0.044053,wireless,-1.049297,-0.0035,-0.015
1,670471,30685546,train,0.005878,and,-0.843256,-0.003780,and,-1.049297,-0.0028,-0.005
2,670472,30685546,train,-0.000972,continuous,-0.843256,-0.004581,continuous,-1.049297,-0.0023,-0.008
3,670473,30685546,train,-0.000972,monitoring,-0.843256,0.002719,monitoring,-1.049297,-0.0031,-0.002
4,670474,30685546,train,-0.000972,of,-0.843256,0.010749,of,-1.049297,-0.0027,-0.012


Flip the sign for the attributions generated by GPT

In [4]:
# def main(start_index: int, num=1000):
from tqdm import tqdm
import json
from pathlib import Path
import workers
import warnings
import pandas as pd
import os

warnings.filterwarnings("ignore")

dfs_by_id = {key: value for key, value in df.groupby("id")}
cols = ["gpt_index_value", "gpt_index_word_value"]

out_dir = Path("results") / "200_pc_per_token_gpt_flipped"
out_dir.mkdir(parents=True, exist_ok=True)
# print("Indexes:")
# print(start_index, end_index-1)
ids = list(sorted(dfs_by_id.keys()))

output = []

# for each instance
for id in tqdm(ids):
    output.append({"id": id})

    # for each explainer
    for col in cols:
        # print(col)
        pc_pos, tokens_pos, pc_neg, tokens_neg = workers.compute_aopc_separate(
            (id, dfs_by_id[id], col)
        )
        df_temp = pd.DataFrame(
            {"index": list(range(len(pc_pos))), "token": tokens_pos, "pc_value": pc_pos}
        )
        df_temp.to_csv(out_dir / f"{id}_{col}_pos.csv", index=False)

        df_temp = pd.DataFrame(
            {"index": list(range(len(pc_neg))), "token": tokens_neg, "pc_value": pc_neg}
        )
        df_temp.to_csv(out_dir / f"{id}_{col}_neg.csv", index=False)

  0%|          | 0/200 [00:00<?, ?it/s]Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set to use cuda
  0%|          | 1/200 [00:14<47:43, 14.39s/it]Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set to use cuda
  1%|          | 2/200 [00:24<40:01, 12.13s/it]Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set to use cuda
  2%|▏         | 3/200 [00:34<36:32, 11.13s/it]Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set to use cuda
  2%|▏         | 4/200 [00:39<27:29,  8.42s/it]Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set to use cuda
Device set to use cuda
  2%|▎         | 5/200 [01:02<45:13, 13.92s/it]Device set to use cuda
Device set to

Calculate AOPC for each article

In [10]:
from tqdm import tqdm
import json
import pandas as pd

import scipy.stats as stats
import numpy as np

from pathlib import Path
import pandas as pd

working_dir = Path(r"f:\rct_gpt_perturbation")

file_path = working_dir / "info.json"

if file_path.exists():
    with file_path.open("r", encoding="utf-8") as f:
        data = json.load(f)
else:
    print(f"Error: {file_path} does not exist in {Path.cwd()}")

ids = data["id"]

out_dir = working_dir / "results" / "200_pc_per_token_gpt_flipped"

cols_mapping = {
    # "sv_value": "SHAP",
    # "ig_value": "IG",
    "gpt_index_value": "GPT-index",
    "gpt_index_word_value": "GPT-token",
}

# df = pd.read_csv(os.path.join("results", "aopc_results.csv"))

df_output_list = []

for id in ids:
    _temp_dict = {"id": id}
    for col in cols_mapping.keys():
        df_pos = pd.read_csv(out_dir / f"{id}_{col}_pos.csv")
        df_neg = pd.read_csv(out_dir / f"{id}_{col}_neg.csv")
        df_all = pd.concat([df_pos, df_neg], ignore_index=True)
        _temp_dict[f"aopc_{col}"] = df_all["pc_value"].mean()
        _temp_dict[f"aopc_pos_{col}"] = df_pos["pc_value"].mean()
        _temp_dict[f"aopc_neg_{col}"] = df_neg["pc_value"].mean()
    df_output_list.append(_temp_dict)

df_output = pd.DataFrame(df_output_list)
df_output.to_csv(working_dir / "results" / "aopc_gpt_flipped.csv", index=False)

Get mean and CI for AOPC

In [11]:
def mean_ci(series, confidence=0.95):
    series = series.dropna()  # Remove NaN values
    if len(series) == 0:  # If empty after dropping NaNs, return "NaN"
        return "NaN (NaN, NaN)"

    mean = np.mean(series)

    if len(series) == 1:  # If only one value, no confidence interval can be calculated
        return f"{mean:.3f}"

    sem = stats.sem(series)  # Standard error of the mean
    ci_range = sem * stats.t.ppf((1 + confidence) / 2.0, len(series) - 1)
    lower = mean - ci_range
    upper = mean + ci_range

    return f"{mean:.3f} ({lower:.3f}, {upper:.3f})"


df_aopc_list = []

for explainer in cols_mapping.keys():
    df_aopc_list.append(
        {
            "Explainer": cols_mapping[explainer],
            "AOPC": mean_ci(df_output[f"aopc_{explainer}"]),
            "AOPC (Positive Tokens)": mean_ci(df_output[f"aopc_pos_{explainer}"]),
            "AOPC (Negative Tokens)": mean_ci(df_output[f"aopc_neg_{explainer}"]),
        }
    )


df_aopc = pd.DataFrame(df_aopc_list)
df_aopc.to_csv(working_dir / "results" / "aopc_table_gpt_flipped.csv", index=False)
display(df_aopc)

,Explainer,AOPC,AOPC (Positive Tokens),AOPC (Negative Tokens)
0,GPT-index,"-0.019 (-0.032, -0.006)","0.032 (0.019, 0.046)","-0.046 (-0.063, -0.028)"
1,GPT-token,"-0.028 (-0.043, -0.014)","0.022 (0.011, 0.034)","-0.050 (-0.070, -0.030)"
